# Lab 4: What a Complete Project Looks Like

**Before you start: go to Runtime → Change runtime type and select GPU.**

This is a short notebook. Its purpose is to show you what a complete, 
minimal deep learning project looks like from start to finish — not just 
training, but the full arc that your project report will need to cover:

1. A clearly defined problem
2. A dataset with known properties
3. A baseline result
4. An improved result
5. An analysis of what works and what does not
6. Results presented clearly

You have seen pieces of this before. Today you see it all together.

**The main event today is the proposal workshop**, not this notebook. 
Read through it, run it, and use the second half of the session 
to finalise your proposal slides and get feedback from an instructor or TA.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
import numpy as np
import zipfile, gdown
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Step 1: Define the problem clearly

A good project has a crisp one-sentence problem statement. Everything else — 
data choices, model choices, evaluation metrics — follows from it.

> **Problem:** Classify images of cats, dogs, and horses into their correct 
> species using a convolutional neural network trained on a small labelled dataset.
>
> **Why it matters:** Automated animal identification is useful in wildlife 
> monitoring, veterinary triage, and livestock management.
>
> **Success criterion:** Validation accuracy above 95% on a held-out test set.

Notice what this statement includes:
- What the input is (images)
- What the output is (species label)
- The approach (CNN, small dataset)
- A concrete, measurable success criterion

Your proposal should have an equivalent statement for your own project.

## Step 2: Know your dataset

In [ ]:
# Load data (same dataset as weeks 2 and 3)
gdown.download('https://drive.google.com/uc?id=1KDMC39ba1MAL83FLLoSVSJY2KOmFR1aj', 'data.zip', quiet=True)
with zipfile.ZipFile('data.zip', 'r') as z:
    z.extractall('.')
data_dir = Path('data')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(data_dir / 'train', transform=train_transform)
val_ds   = datasets.ImageFolder(data_dir / 'val',   transform=val_transform)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)

CLASS_NAMES = train_ds.classes
print(f'Classes:    {CLASS_NAMES}')
print(f'Train size: {len(train_ds)}')
print(f'Val size:   {len(val_ds)}')

## Step 3: Establish a baseline

A baseline is the simplest model you can train. It tells you two things:
- Whether your pipeline is working at all
- How much room for improvement there is

Here, baseline = a small CNN trained from scratch, no pretrained weights.

In [ ]:
def make_baseline_cnn(num_classes):
    """A small CNN trained from scratch — our baseline."""
    return nn.Sequential(
        nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((4, 4)),
        nn.Flatten(),
        nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(256, num_classes)
    )

def train_and_evaluate(model, train_dl, val_dl, epochs=10, lr=1e-3):
    """Train a model and return its history."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history   = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

    for epoch in range(1, epochs + 1):
        model.train()
        t_loss = t_correct = t_total = 0
        for imgs, lbls in train_dl:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, lbls)
            loss.backward()
            optimizer.step()
            t_loss    += loss.item() * imgs.size(0)
            t_correct += (out.argmax(1) == lbls).sum().item()
            t_total   += imgs.size(0)

        model.eval()
        v_loss = v_correct = v_total = 0
        with torch.no_grad():
            for imgs, lbls in val_dl:
                imgs, lbls = imgs.to(device), lbls.to(device)
                out  = model(imgs)
                loss = criterion(out, lbls)
                v_loss    += loss.item() * imgs.size(0)
                v_correct += (out.argmax(1) == lbls).sum().item()
                v_total   += imgs.size(0)

        ta = t_correct / t_total
        va = v_correct / v_total
        history['train_acc'].append(ta)
        history['val_acc'].append(va)
        history['train_loss'].append(t_loss / t_total)
        history['val_loss'].append(v_loss / v_total)
        print(f'Epoch {epoch:2d}/{epochs}  train_acc={ta:.3f}  val_acc={va:.3f}')

    return model, history


print('Training baseline (small CNN from scratch)...')
baseline_model, baseline_history = train_and_evaluate(
    make_baseline_cnn(len(CLASS_NAMES)), train_dl, val_dl, epochs=10
)
baseline_val_acc = max(baseline_history['val_acc'])
print(f'\nBaseline best val accuracy: {baseline_val_acc:.3f}')

## Step 4: Improve on the baseline

Now we use transfer learning — a pretrained ResNet-18 instead of training from scratch. 
This is the most impactful single change you can make for most image classification problems.

In [ ]:
def make_transfer_model(num_classes):
    """ResNet-18 pretrained on ImageNet, head replaced for our task."""
    m    = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

print('Training improved model (transfer learning)...')
transfer_model, transfer_history = train_and_evaluate(
    make_transfer_model(len(CLASS_NAMES)), train_dl, val_dl, epochs=10
)
transfer_val_acc = max(transfer_history['val_acc'])
print(f'\nTransfer learning best val accuracy: {transfer_val_acc:.3f}')
print(f'Improvement over baseline: +{(transfer_val_acc - baseline_val_acc)*100:.1f}%')

## Step 5: Analyse the results

Numbers alone are not enough. A good project report shows *where* the model 
succeeds and fails, not just its overall accuracy.

In [ ]:
# Loss curves — side by side for easy comparison
epochs = range(1, 11)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in zip(
    axes, ['val_acc', 'val_loss'], ['Validation accuracy', 'Validation loss']
):
    ax.plot(epochs, baseline_history[metric],  label='Baseline (from scratch)', linestyle='--')
    ax.plot(epochs, transfer_history[metric],  label='Transfer learning')
    ax.set_xlabel('Epoch')
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Baseline vs transfer learning', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix for the transfer learning model
transfer_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in val_dl:
        preds = transfer_model(imgs.to(device)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(lbls.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(all_labels, all_preds),
    display_labels=CLASS_NAMES
).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion matrix — transfer learning model')
plt.tight_layout()
plt.show()

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

## Step 6: Present results clearly

A results table is the standard way to report model performance in a paper. 
It should show enough information for the reader to understand what was tried 
and what worked. Here is a template you can adapt for your project:

In [ ]:
# A publication-style results summary table
# Adapt this for your own project

results = [
    # (Model,                   Pretrained,  Params,    Val Acc)
    ('Small CNN (baseline)',     'No',        '630K',    f'{baseline_val_acc:.3f}'),
    ('ResNet-18 (transfer)',     'ImageNet',  '11.2M',   f'{transfer_val_acc:.3f}'),
]

print(f'{'Model':<30} {'Pretrained':<12} {'Params':<10} {'Val Acc':<10}')
print('-' * 64)
for row in results:
    print(f'{row[0]:<30} {row[1]:<12} {row[2]:<10} {row[3]:<10}')

print()
print('Key finding: transfer learning from ImageNet gives a substantial')
print('improvement over training from scratch on this small dataset.')

## The iterative workflow

The arc above — problem, data, baseline, improvements, analysis, results — 
describes the *structure* of a project. But it does not describe how you 
actually get from a baseline to good results. That process is iterative, 
and it has a specific shape that distinguishes strong projects from weak ones.

Here is the workflow you should follow from week 5 until submission:

```
┌─────────────────────────────────────────────────────────┐
│                   INITIAL STAGE                         │
│   Acquire data → set up baseline model → train & test   │
└─────────────────────┬───────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────┐
│                  ONE ITERATION                          │
│                                                         │
│  1. CLARIFY  — list all current issues with your setup  │
│               (low accuracy, overfitting, slow, etc.)   │
│                      │                                  │
│  2. PRIORITISE — pick the single most important issue   │
│                      │                                  │
│  3. ANALYSE  — find possible causes of that issue       │
│                      │                                  │
│  4. HYPOTHESISE — design one or more testable solutions │
│                      │                                  │
│  5. EVALUATE — run the experiment, measure the result   │
│                      │                                  │
│         repeat ◄─────┘  (go back to step 1)            │
└─────────────────────────────────────────────────────────┘
```

Each pass through this loop is one experiment in your report.

### Why this matters for your report

A strong report does not just describe *what* you did — it shows 
the reasoning behind each decision. For every experiment, you should 
be able to answer:

- **Why did you try this?** (what issue were you addressing?)
- **What did you expect to happen?** (your hypothesis)
- **What actually happened?** (your result)
- **What does that tell you?** (your interpretation)

If you cannot answer all four questions for an experiment, it is not 
a motivated experiment — it is random tinkering. Random tinkering 
cannot be written up as a strong report.

### A concrete example

**Weak:** *'We tried adding dropout and accuracy improved from 78% to 83%.'*

**Strong:** *'Our loss curves showed a 12-point gap between training and 
validation accuracy, indicating overfitting. We hypothesised that adding 
dropout (p=0.5) to the classifier head would reduce this gap by preventing 
co-adaptation of features. After adding dropout, the training/validation 
gap reduced from 12 to 4 points and validation accuracy improved from 
78% to 83%, confirming our hypothesis.'*

Both describe the same experiment. Only the second one demonstrates understanding.

### What this means for your proposal

Your proposal should already be written in the language of this workflow. 
Instead of 'we will train a ResNet-18', write:

> *'Our baseline will be a fine-tuned ResNet-18. We expect the main 
> challenge to be overfitting given our small dataset (~300 images per class). 
> Our planned experiments will address this through: (1) data augmentation, 
> (2) weight decay tuning, and (3) comparison of backbone sizes.'*

That is a proposal that demonstrates you have already thought about the 
iterative process — not just the end goal.

## What this means for your project

Every good project report follows this same arc:

1. **Problem statement** — what are you trying to do and why?
2. **Dataset** — what data did you use, and what are its properties?
3. **Baseline** — what is the simplest thing you tried first?
4. **Improvements** — what did you try next, and did it help?
5. **Analysis** — where does your model succeed and fail?
6. **Results table** — a clean summary that lets the reader compare methods

Your proposal does not need results yet — but it should describe steps 1–3 
clearly and give a plan for steps 4–6, written in the language of the 
iterative workflow above.

---

## Now: prepare for the proposal workshop

For the rest of the session:

1. Finish or polish your 1–2 proposal slides (see the slide template)
2. Wait for an instructor or TA to come to your group
3. Walk them through your slides — 5 minutes max
4. Listen to feedback and take notes
5. Use any remaining time to improve your proposal before submission

**Your proposal is due after this session.** The feedback you get today 
should feed directly into it.